In [ ]:
!git clone https://github.com/jeminanda/hometrainer.git

In [ ]:
!pip install kagglehub

In [2]:
import sys
from pathlib import Path
import numpy as np

# 1. 프로젝트 루트 경로 설정 (노트북 위치에 맞게 필요시 Path("..") 조절)
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# 2. 추출기 모듈 불러오기
from hometrainer.src.pose_extraction.pose_extractor import(
    BlazePoseExtractor,
    ExtractionConfig,
    PoseNotDetectedError
)

print("✅ BlazePoseExtractor 로드 완료!")

ModuleNotFoundError: No module named 'hometrainer'

In [2]:
# -------------------------------------------------------------
# 설정 (ExtractionConfig)
# -------------------------------------------------------------
config = ExtractionConfig(
    model_path="models/pose_landmarker_full.task", # .task 모델 경로
    include_z=True,            # Z 좌표 포함 여부 (x, y, z, visibility = 4채널)
    target_fps=None,           # None인 경우 원본 영상 FPS 사용
    min_detected_ratio=0.1     # 최소 포즈 검출 비율 (10%)
)

# 입력 영상 및 저장할 .npy 파일 경로 설정
video_path = "data/raw/squat_001.mp4"
output_npy_path = "data/raw/squat_001_keypoints.npy"

# -------------------------------------------------------------
# 키포인트 추출 실행
# -------------------------------------------------------------
try:
    with BlazePoseExtractor(config) as extractor:
        print(f"🎥 [{video_path}] 키포인트 추출 시작...")
        keypoints, meta = extractor.extract_from_video(video_path)

    print("\n✅ 추출 성공!")
    print(f"• Keypoints Shape (T, J, C): {keypoints.shape}")
    print(f"• Metadata: {meta}")

    # .npy 파일 저장
    save_path = Path(output_npy_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(save_path, keypoints)
    print(f"💾 저장 완료: {save_path.resolve()}")

except PoseNotDetectedError as e:
    print(f"❌ 포즈 검출 실패: {e}")
except FileNotFoundError as e:
    print(f"❌ 파일을 찾을 수 없습니다: {e}")
except Exception as e:
    print(f"❌ 에러 발생: {e}")

NameError: name 'ExtractionConfig' is not defined

In [3]:
raw_dir = Path("data/raw")
video_extensions = ["*.mp4", "*.avi", "*.mov"]

# 비디오 파일 목록 검색
video_files = []
for ext in video_extensions:
    video_files.extend(list(raw_dir.glob(ext)))

print(f"🔍 총 {len(video_files)}개의 비디오 파일 발견\n")

# 배치 추출 실행
with BlazePoseExtractor(config) as extractor:
    for vid_path in sorted(video_files):
        # build_dataset.py의 규칙에 맞게 저장 파일명 생성 ({exercise}_{id}_keypoints.npy)
        output_file = vid_path.parent / f"{vid_path.stem}_keypoints.npy"
        
        try:
            print(f"🔄 처리 중: {vid_path.name} -> {output_file.name}")
            keypoints, meta = extractor.extract_from_video(vid_path)
            
            # .npy 저장
            np.save(output_file, keypoints)
            print(f"   ↳ 완료! Shape: {keypoints.shape}, 검출률: {meta['detected_ratio']:.1%}")
            
        except PoseNotDetectedError as e:
            print(f"   ↳ ⚠️ 스킵 (포즈 미검출): {e}")
        except Exception as e:
            print(f"   ↳ ❌ 실패: {e}")

print("\n🎉 모든 영상 처리 완료!")

🔍 총 7개의 비디오 파일 발견



NameError: name 'BlazePoseExtractor' is not defined

In [4]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# hometrainer/src/preprocessing 경로에서 모듈 불러오기
from src.preprocessing import (
    normalize_landmarks,
    calculate_angle,
    slice_repetitions,
    animate_skeleton_2d
)

# 데이터셋 경로 지정
RAW_DATA_DIR = os.path.join("hometrainer", "data", "raw")

In [ ]:
import numpy as np
from src.preprocessing import normalize_landmarks  # 경로에 맞춰 import

# 1. 파일 경로 지정 및 .npy 파일 로드
npy_path = "data/raw/sample_exercise_keypoints1.npy"
sample_keypoints_npy = np.load(npy_path)

# 2. 로드된 데이터 Shape 확인 (Shape: (T, J, C))
print(f"로드된 원본 키포인트 Shape: {sample_keypoints_npy.shape}")

# 3. src.preprocessing.normalize_landmarks 모듈 적용
normalized_landmarks_array = normalize_landmarks(sample_keypoints_npy)

# 4. 결과 출력
print(f"정규화된 3D 랜드마크 shape: {normalized_landmarks_array.shape}")
print(f"정규화된 3D 랜드마크 sample: {normalized_landmarks_array[15]}")

In [ ]:
# notebook에서 matplotlib 애니메이션을 표시하기 위한 설정 (선택사항)
# %matplotlib notebook # 또는 %matplotlib inline (애니메이션 표시에 jshtml 방식 사용 시)

import matplotlib as mpl

# Notebook에서 jshtml 방식으로 애니메이션을 볼 때 필요한 설정 (가끔 주석 처리해야 작동하기도 함)
# mpl.rc('animation', html='jshtml')

# Cell 3에서 정규화 완료된 'normalized_landmarks_array' 사용

# 1. 시각화할 프레임 시퀀스 선택 (예: 스쿼트 동작이 포함된 연속된 90프레임)
# 실제 데이터의 시퀀스 ID 등을 기준으로 끊어서 사용해야 함
squat_idx = 0
animation_sequence = normalized_landmarks_array[squat_idx:squat_idx+100] # 90프레임 시퀀스 예시

# 2. 3D 스켈레톤 애니메이션 생성
print("Squat Animation Generating...")
squat_ani = animate_skeleton_2d(
    animation_sequence, 
    save_path="tests/squat_animation.gif", # GIF로 저장하려면 주석 해제 (저장 안 하려면 None)
    title=f"Squat Normalized 3D Skeleton Animation"
)

# Notebook에서 애니메이션 객체를 반환하여 표시
# 만약 `plot_skeleton_3d` 내부에서 `plt.show()`를 호출하지 않았고, 
# Notebook 설정을 마쳤다면 아래 줄만으로 애니메이션이 재생됨
squat_ani

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# 프로젝트 루트 경로를 sys.path에 추가 (ipynb 위치에 따라 조정)
# 현재 노트북이 notebooks/ 폴더 안에 있다면 '..'으로 루트를 지정합니다.
project_root = Path(".").resolve()  # 필요시 Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.pose_extraction import BlazePoseExtractor, ExtractionConfig, PoseNotDetectedError

# -------------------------------------------------------------
# 1. 설정 생성 (3D 좌표 x, y, z, visibility 모두 포함 시 include_z=True)
# -------------------------------------------------------------
config = ExtractionConfig(
    model_path="models/pose_landmarker_full.task",
    include_z=False,            # (x, y, visibility) -> shape (T, 33, 3)
    target_fps=None,            # None이면 원본 FPS 유지
    min_detected_ratio=0.1      # 전체 프레임 중 최소 10%는 검출되어야 함
)

input_video_path = "data/raw/Pushup2.mp4"  # 추출할 영상 경로
output_npy_path = "data/raw/sample_exercise_keypoints2.npy"

# -------------------------------------------------------------
# 2. 키포인트 추출 실행
# -------------------------------------------------------------
try:
    with BlazePoseExtractor(config) as extractor:
        print(f"[{input_video_path}] 키포인트 추출 시작...")
        keypoints, meta = extractor.extract_from_video(input_video_path)

    print("\n✅ 추출 완료!")
    print(f"- 키포인트 배열 Shape (T, J, C): {keypoints.shape}")
    print(f"- 메타데이터: {meta}")

    # -------------------------------------------------------------
    # 3. 데이터 (.npy) 저장
    # -------------------------------------------------------------
    save_path = Path(output_npy_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(save_path, keypoints)
    print(f"- .npy 저장 완료: {save_path.resolve()}")
except PoseNotDetectedError as e:
    print(f"❌ 포즈 검출 실패: {e}")
except FileNotFoundError as e:
    print(f"❌ 파일을 찾을 수 없습니다: {e}")

In [ ]:
# 저장한 .npy 불러오기 검증
loaded_kps = np.load(output_npy_path)

print(f"로드된 데이터 Shape: {loaded_kps.shape}")

# 첫 번째 프레임(T=0)의 코(Nose, Index 0), 왼쪽 무릎(Left Knee, Index 25) 좌표 출력
# BlazePose 키포인트 Index: 0(Nose), 11/12(Shoulders), 23/24(Hips), 25/26(Knees), 27/28(Ankles)
first_frame = loaded_kps[0]
print("\n[프레임 0 주요 관절 좌표 (x, y, visibility)]")
print(f"- Nose (0): {first_frame[0]}")
print(f"- Left Knee (25): {first_frame[25]}")

# 시간(T) 흐름에 따른 오른쪽 무릎(Right Knee, Index 26) Y 좌표 변화 시각화 (스쿼트/운동 템포 확인)
plt.figure(figsize=(10, 4))
plt.plot(loaded_kps[:, 26, 1], label="Right Knee Y-coord (Normalized)", color="red")
plt.gca().invert_yaxis()  # 이미지 좌표계는 위쪽이 0이므로 Y축 반전
plt.title("Right Knee Y Coordinate Trajectory Across Frames")
plt.xlabel("Frame Index (T)")
plt.ylabel("Y Position")
plt.grid(True)
plt.legend()
plt.show()

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# 1. 프로젝트 루트 경로 지정 (노트북이 notebooks/ 내부에 있을 경우 Path("..") 로 설정)
project_root = Path(".").resolve()  # 필요 시 Path("../").resolve()

# 2. build_dataset 파이프라인 모듈 임포트
from src.preprocessing.build_dataset import build_dataset, process_source_file

print("✅ 모듈 로드 완료!")

✅ 모듈 로드 완료!


In [2]:
# 경로 및 매개변수 설정
RAW_DIR = project_root / "data" / "raw"
OUTPUT_DIR = project_root / "data" / "processed"
TARGET_LENGTH = 100  # 정규화할 타겟 프레임 수

# 데이터셋 전처리 파이프라인 실행
print("🚀 데이터셋 전처리 빌드 시작...")
build_dataset(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    manifest_path=None,  # 특정 manifest.csv 사용 시 Path("path/to/manifest.csv")
    target_length=TARGET_LENGTH,
)

🚀 데이터셋 전처리 빌드 시작...
처리 대상 파일 수: 3
[OK] pushup_001_keypoints.npy: rep 105개 처리 완료
[OK] pushup_002_keypoints.npy: rep 1개 처리 완료
[OK] pushup_003_keypoints.npy: rep 6개 처리 완료

완료: rep 112개 (성공 파일 3개 / 실패 0개)
rep_sequences shape: (112, 100, 33, 3)
저장 위치: C:\Users\kccistc\Desktop\workspace\hometrainer\data\processed


In [3]:

# 1. 생성된 파일 확인
rep_seqs = np.load(OUTPUT_DIR / "rep_sequences.npy")
rep_index_df = pd.read_csv(OUTPUT_DIR / "rep_index.csv")

with open(OUTPUT_DIR / "build_log.json", "r", encoding="utf-8") as f:
    build_log = json.load(f)

print("===== 📊 전처리 결과 데이터 요약 =====")
print(f"• 생성된 Rep 시퀀스 Shape : {rep_seqs.shape}")
# shape 출력 예시: (전체 Rep 수, 100, 33, C)

print(
    f"• 성공 파일 수 : {len(build_log['success'])}개 / 실패 파일 수 : {len(build_log['failed'])}개"
)

# 2. 실패한 파일 로그 출력 (있을 경우)
if build_log["failed"]:
    print("\n⚠️ 실패 로그 예시:")
    for failed_info in build_log["failed"]:
        print(f" - 파일: {failed_info['file']} | 원인: {failed_info['reason']}")

# 3. Rep 인덱스 데이터프레임 미리보기
print("\n===== 📝 rep_index.csv 상위 5개 행 =====")
display(rep_index_df.head())

===== 📊 전처리 결과 데이터 요약 =====
• 생성된 Rep 시퀀스 Shape : (112, 100, 33, 3)
• 성공 파일 수 : 3개 / 실패 파일 수 : 0개

===== 📝 rep_index.csv 상위 5개 행 =====


,video_id,exercise,rep_idx,seq_index
0,1,pushup,0,0
1,1,pushup,1,1
2,1,pushup,2,2
3,1,pushup,3,3
4,1,pushup,4,4
